# 01 · Structure survey — PSMA / GCPII (UniProt Q04609)

**Milestone 1.** This is the refactored version of `PDB_structure_pipeline.ipynb`. The calculations now live in the importable package `src/psma_pipeline/`; this notebook only orchestrates, explains and plots.

**What it does**
1. Finds every PDB entry mapped to the UniProt accession and cross-checks the list with PDBe (SIFTS) and UniProt.
2. Extracts the method, resolution, constructs, missing residues, ligands and ions, glycans, biological assemblies, validation metrics, binding sites, active-site B-factors, structural variability and entrance-lid states.
3. Writes the survey workbook, with the same 14 sheets as before plus new sheets, CSV/Parquet tables and a reproducibility manifest.

**How this milestone runs** (brief §21)

| Step | Purpose | Stops if |
|---|---|---|
| A | Unit tests | any test fails |
| B | Small validation set, then sanity checks | a **critical** check FAILs (numbering, assembly, active site, lid calibration) |
| C | Full survey, then sanity checks | – (problems are reported) |
| D | Comparison with your previous workbook | – (differences are listed) |

Settings live in `config/*.yaml`, not in this notebook. To change a threshold, edit the YAML or pass `overrides=` below.

## 0 · Setup

In **Colab**, the first cell clones the repository and installs the package. Locally, run the notebook from inside the repository after `pip install -e .`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/victorolatunde23-dev/psma_structural_modelling.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not Path("psma_structural_modelling").exists() and not Path("src/psma_pipeline").exists():
    subprocess.run(["git", "clone", "-q", REPO], check=True)   # private repo: use https://<token>@github.com/...
if Path("psma_structural_modelling").exists():
    os.chdir("psma_structural_modelling")
if Path("../src/psma_pipeline").exists():                       # opened from notebooks/
    os.chdir("..")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, str(Path("src").resolve()))
print("project root:", Path.cwd())

**Storage on Google Drive.** Colab deletes its local disk (`/content`) whenever the runtime restarts. All notebooks therefore save downloads and results to one Google Drive folder, `MyDrive/psma_structural_modelling/`, so:

- notebook 02 and later notebooks can read notebook 01's results;
- nothing is downloaded twice.

Colab asks once for permission to access Drive. Set `USE_DRIVE = False` to use the temporary disk instead.

In [ ]:
from psma_pipeline.colab import storage_overrides
USE_DRIVE = True
overrides = storage_overrides(USE_DRIVE)

## 1 · Configuration, logging and available software

In [ ]:
import pandas as pd
import psma_pipeline
from psma_pipeline import load_config, setup_logging
from psma_pipeline.manifest import detect_software

setup_logging()
cfg = load_config(overrides=overrides or None)
print("pipeline", psma_pipeline.__version__, "| target", cfg.accession, "|", cfg.get("target.name"))
print("config files:", *cfg.sources, sep="\n  ")
software = detect_software()
display(software[software["Kind"] == "python package"].set_index("Software")[["Available", "Version"]].T)

## A · Unit tests

These cover SIFTS label_seq → UniProt mapping, author/label chains, assembly choice and symmetry-generated dimer counting, contacts and Zn coordination, the lid classifier, ranking with missing values, configuration validation, and the preservation of every original column.

Tests that need gemmi run here in Colab, where gemmi is installed. Network tests are opt-in (`PSMA_NETWORK_TESTS=1`).

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "Unit tests failed: fix before running the survey."

## B · Validation set

Selecting the set uses only metadata (one cached GraphQL call per 25 entries), so no structures are downloaded yet. The categories follow brief §20:

- calibration structures **3BHX** (closed lid) and **2XEG** (open lid; its ligand sits in the arene-binding site), **2OOT** (lid not modelled: must be called *disordered*, never open), and the previous top template **9S7U** (`validation_set.fixed_ids`);
- the best high-resolution dimer with 2 chains in the ASU;
- an ASU monomer whose biological assembly is a dimer;
- a ligand-bound entry;
- an apo entry;
- a cryo-EM entry;
- optionally, another disordered-lid entry picked from your **previous workbook**. Set its path below.

In [ ]:
from psma_pipeline.sanity import select_validation_set, validation_ids, run_sanity_checks, critical_failures, sanity_overview
from psma_pipeline.survey import fetch_entry_metadata, run_survey, save_survey

PREVIOUS_WORKBOOK = cfg.get("validation_set.previous_workbook")   # e.g. "Q04609_structures.xlsx" from the old notebook
all_ids, meta = fetch_entry_metadata(cfg)
print(f"{len(all_ids)} PDB entries mapped to {cfg.accession}")
selection = select_validation_set(meta, cfg, PREVIOUS_WORKBOOK)
selection

In [ ]:
val_ids = validation_ids(selection, meta)
val = run_survey(cfg, ids=val_ids)
val_sanity = run_sanity_checks(val.tables, cfg)
display(sanity_overview(val_sanity))
cols = ["PDB ID", "Method", "Resolution (Å)", "Target copies in ASU", "Target copies in assembly",
        "Monomer / dimer", "Assembly choice reason", "Entrance lid state", "Warnings"]
display(val.tables["Summary"][[c for c in cols if c in val.tables["Summary"]]])

### Gate

The full run continues only if **no critical check FAILs**. A FAIL is not overridden here. Read the detail, fix the cause (usually the configuration or a numbering edge case), and re-run.

A WARNING is recorded and does not block the run. An example is an engineered E424A mutant whose active-site residue is excluded from the B-factor summaries.

In [ ]:
crit = critical_failures(val_sanity)
display(val_sanity[val_sanity["Status"].isin(["FAIL", "WARNING"])])
val_paths = save_survey(cfg, val, extra_sheets={"SanityChecks": val_sanity, "ValidationSet": selection},
                        tag="validation")
print("validation outputs:", val_paths["workbook"])
assert crit.empty, "Critical sanity checks failed on the validation set: full run stopped."

## C · Full survey

Every entry is processed with the same code. One entry failing never stops the survey: the failure is recorded in `Summary → Warnings` and in the `RunIssues` sheet.

In [ ]:
res = run_survey(cfg)
sanity = run_sanity_checks(res.tables, cfg)
display(sanity_overview(sanity))
paths = save_survey(cfg, res, extra_sheets={"SanityChecks": sanity, "ValidationSet": selection})
print("workbook:", paths["workbook"]); print("manifest:", paths["manifest"]); print("tables:", paths["tables_dir"])
T = res.tables

### Warnings and failed entries

In [ ]:
issues = res.issues.to_frame()
display(issues["Status"].value_counts().to_frame("issues"))
display(issues[issues["Status"].isin(["FAIL", "WARNING"])].head(50))
display(sanity[sanity["Status"].isin(["FAIL", "WARNING"])].head(50))

## Results at a glance

In [ ]:
s = T["Summary"]
display(s["Monomer / dimer"].value_counts().to_frame("entries"))
display(s["Entrance lid state"].value_counts(dropna=False).to_frame("entries") if "Entrance lid state" in s else None)
cols = ["PDB ID", "Rank", "Method", "Resolution (Å)", "Clashscore", "Score (0-1, higher = better)",
        "Score explanation"]
display(T["TemplateRanking"][[c for c in cols if c in T["TemplateRanking"]]].head(15))

### Plots

1. Resolution over time, by experimental method.
2. Geometry quality against resolution, with the top-10 templates highlighted.
3. Active-site normalised B: blue is well ordered, red is mobile, grey is not modelled.
4. Entrance-lid states: PC1 of the superposed lid coordinates against helicity.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
d = s.dropna(subset=["Resolution (Å)"]).copy()
d["Year"] = pd.to_datetime(d["Release date"], errors="coerce").dt.year
for m, g in d.groupby("Method"):
    axes[0].scatter(g["Year"], g["Resolution (Å)"], label=m, alpha=0.75)
axes[0].invert_yaxis(); axes[0].set(xlabel="Release year", ylabel="Resolution (Å)", title="Resolution over time")
axes[0].legend(fontsize=8)
if "Clashscore" in d:
    top = set(T["TemplateRanking"]["PDB ID"].head(10))
    c = d.dropna(subset=["Clashscore"])
    axes[1].scatter(c["Resolution (Å)"], c["Clashscore"], alpha=0.5, label="all")
    t = c[c["PDB ID"].isin(top)]
    axes[1].scatter(t["Resolution (Å)"], t["Clashscore"], color="crimson", label="top-10 (global mode)")
    for _, r in t.iterrows():
        axes[1].annotate(r["PDB ID"], (r["Resolution (Å)"], r["Clashscore"]), fontsize=7)
    axes[1].set(xlabel="Resolution (Å)", ylabel="Clashscore", title="Geometry vs resolution"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
a = T["ActiveSite"]
if len(a) and "Normalised B (z)" in a:
    a = a[a["Matches UniProt"].fillna(True).astype(bool)] if "Matches UniProt" in a else a
    hm = a.pivot_table(index="PDB ID", columns="UniProt residue", values="Normalised B (z)", aggfunc="mean")
    order = sorted(hm.columns, key=lambda r: int(r[1:]))
    hm = hm[order].loc[[p for p in T["TemplateRanking"]["PDB ID"] if p in hm.index][:40]]
    fig, ax = plt.subplots(figsize=(1 + 0.45 * len(order), 0.25 * len(hm) + 1.5))
    cmap = plt.get_cmap("coolwarm").copy(); cmap.set_bad("lightgrey")
    im = ax.imshow(np.ma.masked_invalid(hm.values), cmap=cmap, vmin=-2, vmax=2, aspect="auto")
    ax.set_xticks(range(len(order)), order, rotation=90, fontsize=8); ax.set_yticks(range(len(hm)), hm.index, fontsize=7)
    ax.set_title("Active-site normalised B (z), top-ranked 40 entries; grey = not modelled")
    fig.colorbar(im, ax=ax, shrink=0.6, label="z"); plt.tight_layout(); plt.show()

In [ ]:
lid = T["EntranceLid"]
if len(lid) and "Lid PC1 score" in lid:
    d = lid.dropna(subset=["Lid PC1 score"])
    colours = {"open": "tab:orange", "closed": "tab:blue"}
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for st_, g in d.groupby("Lid state"):
        axes[0].scatter(g["Lid PC1 score"], g["Lid helicity (CA i,i+3)"], label=st_, color=colours.get(st_, "grey"), alpha=0.75)
    if "Calibration label (literature)" in d:
        for _, r in d.dropna(subset=["Calibration label (literature)"]).iterrows():
            axes[0].annotate(f"{r['PDB ID']} ({r['Calibration label (literature)']})",
                             (r["Lid PC1 score"], r["Lid helicity (CA i,i+3)"]), fontsize=8)
    axes[0].axhline(cfg.lid["classifier"]["helix_fraction_min"], ls="--", lw=0.8, color="k")
    axes[0].set(xlabel="Lid PC1 score", ylabel="Lid helicity", title=f"Lid states (Ashman D = {res.lid_info.get('ashman_D')})")
    axes[0].legend(fontsize=8)
    axes[1].hist(d["Lid PC1 score"], bins=30, color="tab:grey")
    axes[1].set(xlabel="Lid PC1 score", ylabel="Chains", title="Bimodal = two discrete states")
    plt.tight_layout(); plt.show()
display(T["LidVsLigand"])

## D · Comparison with the previous milestone

Point `PREVIOUS_WORKBOOK` at the workbook from the original notebook (or from the previous milestone). The comparison lists:

- added and removed sheets and columns;
- row-count changes;
- value differences on rows matched by PDB ID, chain and component.

**Expected differences in Milestone 1** (see `docs/CHANGES.md`):
- new columns and sheets;
- `Waters in contact` now counts unique waters;
- an extra Summary warning when an entry has no SIFTS mapping;
- the active-site mean normalised B excludes engineered substitutions;
- `Notes` has more rows.

Any other value difference should be investigated before Milestone 2.

In [ ]:
from psma_pipeline.excel_output import compare_workbooks
if PREVIOUS_WORKBOOK and Path(PREVIOUS_WORKBOOK).exists():
    cmp = compare_workbooks(PREVIOUS_WORKBOOK, paths["workbook"])
    cmp.to_csv(cfg.outputs_dir / "milestone1_comparison.csv", index=False)
    display(cmp[cmp["Kind"] != "cell"])
    display(cmp[cmp["Kind"] == "cell"].head(50))
else:
    print("Set PREVIOUS_WORKBOOK (section B) to compare with the original notebook's workbook.")

## Milestone checklist

The living checklist is `docs/CHECKLIST.md`. Milestone 1 status:

| Component | Status |
|---|---|
| Modules with docstrings, type hints, controlled exceptions | ✔ implemented, tested offline |
| YAML configuration (target, active site, regions, ligand classes, ranking) | ✔ |
| Logging and RunIssues sheet (no silent failures) | ✔ |
| Identifier and status columns on per-row tables | ✔ |
| Reproducibility manifest (JSON + sheet), software detection | ✔ |
| Data dictionary, status colouring, CSV/Parquet output | ✔ |
| Sanity checks and validation-set selection | ✔ |
| Workbook comparison | ✔ |
| NMR `model_policy: all` | partial: falls back to model 1 with a warning (M2) |